# Stage 4M.1 — Tie-Preserving Missingness-Burden Correction

This notebook corrects one descriptive defect in Stage 4 and the Stage 4 missingness addendum. The earlier expression `qcut(missing_fraction.rank(method='first'), 3)` split identical missingness values according to source-row order. Because source rows were outcome-ordered, the reported burden tertiles were invalid.

This correction performs **no model fitting, tuning, predictor selection, or threshold optimization**. It uses fixed, tie-preserving categories aligned with the already frozen 5% near-complete threshold:

- `complete`: missing fraction = 0;
- `>0–5%`: 0 < missing fraction ≤ 0.05;
- `>5%`: missing fraction > 0.05.

It regenerates corrected prevalence summaries and missingness-group performance using saved patient-averaged OOF predictions. Stage 3B remains primary. `OnlyLargeJoints` remains excluded.

In [ ]:
# 1. Imports and deterministic settings
from __future__ import annotations
import hashlib,json,os,platform,subprocess,sys
from pathlib import Path
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import yaml
from IPython.display import display,Markdown
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score,log_loss,brier_score_loss
MASTER_SEED=20261020;BOOTSTRAPS=2000;MIN_EVENTS=20
OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
np.random.seed(MASTER_SEED);pd.set_option('display.max_columns',150);pd.set_option('display.width',220)
mpl.rcParams.update({'figure.dpi':130,'savefig.dpi':400,'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
sns.set_theme(style='whitegrid',context='paper',palette='colorblind')
print('Python',sys.version.split()[0],'|',platform.platform())

In [ ]:
# 2. Project discovery and isolated correction outputs
NOTEBOOK_NAME='04M1_tie_preserving_missingness_correction.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd();COLAB_PROJECT=Path.cwd()
env=os.environ.get('RA_PROJECT_DIR');candidates=([Path(env)] if env else [])+[ENV_PROJECT,COLAB_PROJECT,Path.cwd(),Path.cwd()/'ra_analysis',Path.cwd()/'stage2_outputs']
PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
if PROJECT_DIR is None:raise FileNotFoundError('Set RA_PROJECT_DIR to the project containing derived_data and modeling_results.')
DERIVED=PROJECT_DIR/'derived_data';S3B=PROJECT_DIR/'modeling_results'/'tables';S3CROOT=PROJECT_DIR/'modeling_results_stage3c'
S3C=next((p for p in [S3CROOT/'tables',S3CROOT/'tables-3c'] if p.exists()),None)
S4=PROJECT_DIR/'modeling_results_stage4'/'tables';S4M=PROJECT_DIR/'modeling_results_stage4_missingness'/'tables'
protocol_candidates=[PROJECT_DIR/'stage4_missingness_tie_correction_protocol.yaml',PROJECT_DIR/'config'/'stage4_missingness_tie_correction_protocol.yaml',Path.cwd()/'stage4_missingness_tie_correction_protocol.yaml',ENV_PROJECT/'stage4_missingness_tie_correction_protocol.yaml']
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
if PROTOCOL_PATH is None:raise FileNotFoundError('Correction protocol YAML not found')
with open(PROTOCOL_PATH,encoding='utf-8') as f:PROTOCOL=yaml.safe_load(f)
OUT=PROJECT_DIR/'modeling_results_stage4_missingness_correction';TABLES=OUT/'tables';FIGURES=OUT/'figures';REPORT=OUT/'report'
for d in [OUT,TABLES,FIGURES,REPORT]:d.mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT_DIR.resolve());print('Correction output:',OUT.resolve())

In [ ]:
# 3. Locate and sign immutable inputs
def first(paths,label):
 p=next((Path(x) for x in paths if Path(x).exists()),None)
 if p is None:raise FileNotFoundError(label)
 return p
def sha(path):
 h=hashlib.sha256();h.update(Path(path).read_bytes());return h.hexdigest()
inputs={
 'stage3b_avg':first([S3B/'patient_averaged_oof_predictions.csv',S3B/'stage3b__patient_averaged_oof_predictions.csv'],'Stage 3B averaged predictions'),
 'stage3c_avg':first([S3C/'patient_averaged_oof_predictions.csv'],'Stage 3C averaged predictions'),
 'selector_avg':first([S4/'nested_selector_patient_averaged_predictions.csv'],'Stage 4 selector predictions'),
 'stage4m_avg':first([S4M/'missingness_sensitivity_patient_averaged_predictions.csv'],'Stage 4M sensitivity predictions'),
 'stage4_audit':first([S4/'stage4_validation_audit.csv'],'Stage 4 audit'),
 'stage4m_audit':first([S4M/'stage4_missingness_validation_audit.csv'],'Stage 4M audit')}
for o in OUTCOMES:
 inputs[f'{o}_data']=DERIVED/f'{o}__primary_eligible.csv';inputs[f'{o}_manifest']=DERIVED/f'{o}__predictor_manifest.csv'
 for p in [inputs[f'{o}_data'],inputs[f'{o}_manifest']]:
  if not p.exists():raise FileNotFoundError(p)
for p in [inputs['stage4_audit'],inputs['stage4m_audit']]:
 a=pd.read_csv(p)
 if not a.status.eq('PASS').all():raise AssertionError(f'Prior audit failed: {p}')
payload={'version':'1.0-tie-preserving-correction','protocol_sha256':sha(PROTOCOL_PATH),'inputs':{k:sha(v) for k,v in inputs.items()},'groups':['complete','>0-5%','>5%'],'bootstraps':BOOTSTRAPS,'seed':MASTER_SEED}
RUN_SIGNATURE=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
with open(OUT/'run_signature.json','w') as f:json.dump({'run_signature':RUN_SIGNATURE,**payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)

## Corrected tie-preserving burden groups

In [ ]:
# 4. Construct groups without splitting ties
strata=[];prevalence=[]
for outcome in OUTCOMES:
 data=pd.read_csv(inputs[f'{outcome}_data']);manifest=pd.read_csv(inputs[f'{outcome}_manifest']);pred=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
 burden=data[pred].isna().mean(axis=1)
 group=pd.cut(burden,[-1e-12,0,.05,1],labels=['complete','>0-5%','>5%'],include_lowest=True).astype(str)
 s=pd.DataFrame({'VisitID':data.VisitID,'outcome':outcome,'y':data[outcome].astype(int),'missing_fraction':burden,'missingness_group_corrected':group});strata.append(s)
 for level in ['complete','>0-5%','>5%']:
  g=s[s.missingness_group_corrected.eq(level)];prevalence.append({'outcome':outcome,'missingness_group_corrected':level,'n':len(g),'events':int(g.y.sum()),'nonevents':int((1-g.y).sum()),'outcome_prevalence':g.y.mean(),'mean_missing_fraction':g.missing_fraction.mean(),'min_missing_fraction':g.missing_fraction.min(),'max_missing_fraction':g.missing_fraction.max()})
strata=pd.concat(strata,ignore_index=True);prevalence=pd.DataFrame(prevalence)
strata.to_csv(TABLES/'corrected_patient_missingness_groups.csv',index=False,encoding='utf-8-sig')
prevalence.to_csv(TABLES/'corrected_outcome_prevalence_by_missingness_group.csv',index=False,encoding='utf-8-sig')
display(prevalence)

In [ ]:
# 5. Verify that no identical burden value is split across groups
mapping_counts=(strata.groupby(['outcome','missing_fraction'],as_index=False).missingness_group_corrected.nunique())
if mapping_counts.missingness_group_corrected.max()!=1:raise AssertionError('Tie-preserving correction failed')
print('PASS: every identical missingness fraction maps to exactly one group.')

## Reconstruct locked OOF prediction sets — evaluation only

In [ ]:
# 6. Assemble the Stage 4 locked models and Stage 4M sensitivity models
s3b=pd.read_csv(inputs['stage3b_avg']);s3c=pd.read_csv(inputs['stage3c_avg']);selector=pd.read_csv(inputs['selector_avg']);sens=pd.read_csv(inputs['stage4m_avg'])
locked=[]
core=s3b[s3b.model.eq('core_ridge')].copy();core['model']='stage3b_core_ridge';locked.append(core[['outcome','model','VisitID','y','probability']])
for outcome in OUTCOMES:
 names=['refined_broad_ridge']+(['xgboost'] if outcome!='NegRFandACCP' else [])
 for name in names:locked.append(s3c.query('outcome==@outcome and model==@name')[['outcome','model','VisitID','y','probability']])
 z=selector[selector.outcome.eq(outcome)].copy();locked.append(z[['outcome','model','VisitID','y','probability']])
locked=pd.concat(locked,ignore_index=True);locked['analysis_set']='stage4_locked_models'
sensitivity=sens[['outcome','model','VisitID','y','probability']].copy();sensitivity['analysis_set']='stage4m_sensitivity_models'
all_predictions=pd.concat([locked,sensitivity],ignore_index=True)
all_predictions=all_predictions.merge(strata,on=['outcome','VisitID','y'],validate='many_to_one')
print(all_predictions.groupby(['analysis_set','outcome','model']).size())

In [ ]:
# 7. Metrics and stable stratified patient bootstrap
EPS=1e-8
def metric_dict(y,p):
 y=np.asarray(y,int);p=np.asarray(p,float);z=logit(np.clip(p,EPS,1-EPS)).reshape(-1,1)
 try:
  q=LogisticRegression(C=np.inf,solver='lbfgs',max_iter=5000).fit(z,y);ci=float(q.intercept_[0]);cs=float(q.coef_[0,0])
 except Exception:ci=cs=np.nan
 return {'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),'log_loss':log_loss(y,p,labels=[0,1]),'brier':brier_score_loss(y,p),'calibration_intercept':ci,'calibration_slope':cs}
def subgroup_row(analysis,outcome,model,level,g):
 n=len(g);events=int(g.y.sum());nonevents=n-events;row={'analysis_set':analysis,'outcome':outcome,'model':model,'missingness_group_corrected':level,'n':n,'events':events,'nonevents':nonevents,'sufficient_for_metrics':events>=MIN_EVENTS and nonevents>=MIN_EVENTS}
 if not row['sufficient_for_metrics']:return row
 point=metric_dict(g.y,g.probability);row.update(point);key=f'{analysis}|{outcome}|{model}|{level}'.encode();seed=MASTER_SEED+int(hashlib.sha256(key).hexdigest()[:8],16)%100000
 rng=np.random.default_rng(seed);y=g.y.to_numpy();p=g.probability.to_numpy();pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0);aucs=[];briers=[]
 for _ in range(BOOTSTRAPS):
  idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)];aucs.append(roc_auc_score(y[idx],p[idx]));briers.append(brier_score_loss(y[idx],p[idx]))
 row['auroc_ci_low']=np.quantile(aucs,.025);row['auroc_ci_high']=np.quantile(aucs,.975)
 row['brier_ci_low']=np.quantile(briers,.025);row['brier_ci_high']=np.quantile(briers,.975)
 row['bootstrap_resamples']=BOOTSTRAPS;return row

In [ ]:
# 8. Corrected missingness-group performance — no refitting
rows=[]
for (analysis,outcome,model,level),g in all_predictions.groupby(['analysis_set','outcome','model','missingness_group_corrected']):rows.append(subgroup_row(analysis,outcome,model,level,g))
subgroup=pd.DataFrame(rows)
locked_corrected=subgroup[subgroup.analysis_set.eq('stage4_locked_models')].copy();sensitivity_corrected=subgroup[subgroup.analysis_set.eq('stage4m_sensitivity_models')].copy()
locked_corrected.to_csv(TABLES/'corrected_stage4_locked_model_missingness_group_performance.csv',index=False,encoding='utf-8-sig')
sensitivity_corrected.to_csv(TABLES/'corrected_stage4m_sensitivity_missingness_group_performance.csv',index=False,encoding='utf-8-sig')
display(locked_corrected);display(sensitivity_corrected)

## Corrected publication figures

In [ ]:
# 9. Corrected prevalence figure
fig,axes=plt.subplots(1,3,figsize=(15,4.8),sharey=False)
order=['complete','>0-5%','>5%']
for ax,outcome in zip(axes,OUTCOMES):
 g=prevalence[prevalence.outcome.eq(outcome)].set_index('missingness_group_corrected').loc[order].reset_index()
 sns.barplot(data=g,x='missingness_group_corrected',y='outcome_prevalence',order=order,color='#2c7fb8',ax=ax)
 for i,r in g.iterrows():ax.text(i,r.outcome_prevalence+.01,f'{int(r.events)}/{int(r.n)}',ha='center',fontsize=9)
 ax.set(title=outcome,xlabel='Tie-preserving missingness group',ylabel='Outcome prevalence',ylim=(0,max(.45,g.outcome_prevalence.max()+.1)))
fig.suptitle('Corrected outcome prevalence by missingness burden');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURES/f'corrected_outcome_prevalence_by_missingness_group.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

In [ ]:
# 10. Corrected locked-model AUROC figure; insufficient groups remain suppressed
p=locked_corrected[locked_corrected.sufficient_for_metrics].copy();p['facet']=p.outcome+' — '+p.missingness_group_corrected;p['label']=p.model
facets=p.facet.unique();fig,axes=plt.subplots(len(facets),1,figsize=(12,max(5,2.3*len(facets))),squeeze=False)
for ax,facet in zip(axes.ravel(),facets):
 g=p[p.facet.eq(facet)].reset_index(drop=True);yy=np.arange(len(g));ax.hlines(yy,g.auroc_ci_low,g.auroc_ci_high,color='#2c7fb8',lw=1.8);ax.scatter(g.auroc,yy,color='#d95f0e',s=28,zorder=3)
 ax.set_yticks(yy,g.label);ax.invert_yaxis();ax.set(title=facet,xlabel='OOF AUROC with 95% patient-bootstrap CI',xlim=(0.3,1.0))
fig.suptitle('Corrected missingness-group performance — evaluation only',y=1.002);fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURES/f'corrected_locked_model_missingness_group_auroc.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

## Correction note and automated audit

In [ ]:
# 11. Machine-readable correction note
note={
 'status':'CORRECTION REQUIRED AND COMPLETED','defect':'rank(method=first) split tied missingness values by source-row order',
 'invalidated_outputs':['prior low/middle/high missingness burden prevalence values','prior Stage 4 missingness_group subgroup performance rows'],
 'unaffected_outputs':['all model fits','all OOF predictions','overall performance','paired comparisons','missingness-only/no-flags/near-complete conclusions','predictor-level missingness summaries'],
 'replacement_groups':{'complete':'0% missing','>0-5%':'>0% to 5% missing','>5%':'>5% missing'},
 'interpretation':'Corrective post hoc descriptive grouping; not prespecified and not used for model selection.',
 'stop_rule':'No further internal model development after this correction.'}
with open(OUT/'correction_note.json','w',encoding='utf-8') as f:json.dump(note,f,indent=2)

In [ ]:
# 12. Automated correction audit
checks=[]
def check(i,d,c,x=''):checks.append({'check_id':i,'description':d,'status':'PASS' if bool(c) else 'FAIL','details':x})
check('C-001','Prior Stage 4 audit passed',pd.read_csv(inputs['stage4_audit']).status.eq('PASS').all())
check('C-002','Prior Stage 4M audit passed',pd.read_csv(inputs['stage4m_audit']).status.eq('PASS').all())
check('C-003','Only permitted outcomes included',set(strata.outcome)==set(OUTCOMES))
check('C-004','OnlyLargeJoints excluded','OnlyLargeJoints' not in strata.outcome.unique())
check('C-005','Every outcome has 820 patients',strata.groupby('outcome').size().eq(820).all())
check('C-006','No patient duplicated within outcome',not strata.duplicated(['outcome','VisitID']).any())
check('C-007','Every tied burden maps to one group',mapping_counts.missingness_group_corrected.max()==1)
check('C-008','All three corrected groups present per outcome',prevalence.groupby('outcome').size().eq(3).all())
check('C-009','Group counts sum to 820',prevalence.groupby('outcome').n.sum().eq(820).all())
check('C-010','Group events equal outcome totals',prevalence.groupby('outcome').events.sum().to_dict()=={'NegRFandACCP':164,'NoHandJointInvolvement':191,'AsymmetricArthritis':190})
check('C-011','No model was fitted in correction',True,'Only saved patient-averaged OOF predictions evaluated')
check('C-012','Every prediction matched one corrected stratum',len(all_predictions)==len(locked)+len(sensitivity))
check('C-013','Small groups suppressed',subgroup.loc[~subgroup.sufficient_for_metrics,'auroc'].isna().all())
check('C-014','Evaluable groups use 2,000 bootstraps',subgroup.loc[subgroup.sufficient_for_metrics,'bootstrap_resamples'].eq(BOOTSTRAPS).all())
check('C-015','All reported probabilities remain bounded',all_predictions.probability.between(0,1).all())
check('C-016','Corrected locked-model table generated',not locked_corrected.empty)
check('C-017','Corrected sensitivity table generated',not sensitivity_corrected.empty)
check('C-018','Correction figures saved as PNG and SVG',all((FIGURES/f'{s}.{e}').exists() for s in ['corrected_outcome_prevalence_by_missingness_group','corrected_locked_model_missingness_group_auroc'] for e in ['png','svg']))
check('C-019','Correction status explicitly post hoc','post hoc' in PROTOCOL['group_definition']['inferential_status'].lower())
check('C-020','Stop rule frozen','stop internal analysis' in PROTOCOL['stop_rule'].lower())
audit=pd.DataFrame(checks);audit.to_csv(TABLES/'missingness_tie_correction_audit.csv',index=False,encoding='utf-8-sig');display(audit)
if not audit.status.eq('PASS').all():raise AssertionError('Correction audit failed')
print('ALL 20 CORRECTION CHECKS PASSED')

In [ ]:
# 13. Artifact registry and summary
art=[]
for folder in [TABLES,FIGURES,OUT]:
 for p in sorted(folder.glob('*')):
  if p.is_file() and p.name!='artifact_registry.csv':art.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(art).drop_duplicates('path').to_csv(TABLES/'artifact_registry.csv',index=False,encoding='utf-8-sig')
summary={'run_signature':RUN_SIGNATURE,'correction_complete':True,'model_refitting':False,'bootstraps':BOOTSTRAPS,'all_20_audits_passed':True,'stop_internal_analysis':True}
with open(OUT/'correction_run_summary.json','w') as f:json.dump(summary,f,indent=2)

In [ ]:
# 14. FINAL CELL — self-contained correction HTML
if os.environ.get('RA_STAGE4M1_EXPORT_MODE')=='1':print('Export subprocess mode: recursive export skipped.')
else:
 candidates=[PROJECT_DIR/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,COLAB_PROJECT/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME]
 notebook=next((p.resolve() for p in candidates if p.exists()),None)
 if notebook is None:raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME}')
 env=os.environ.copy();env['RA_STAGE4M1_EXPORT_MODE']='1';report=REPORT/'Stage4_missingness_tie_correction_report.html'
 cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute','--ExecutePreprocessor.timeout=0','--HTMLExporter.embed_images=True','--output',report.name,'--output-dir',str(REPORT),str(notebook)]
 z=subprocess.run(cmd,env=env,cwd=str(notebook.parent),text=True,capture_output=True)
 if z.stdout:print(z.stdout)
 if z.returncode:print(z.stderr);raise RuntimeError('HTML export failed')
 print('Correction report:',report.resolve())